In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, StringType
import pyspark.sql.functions as sf


In [15]:
#catalog = spark.conf.get("catalog")
catalog = "runescape_dev"

In [16]:
# create path and schema for table
path = f"/Volumes/runescape_dev/00_landing/data_sources/latest_prices/latest_prices_test.json"

In [17]:
schema = StructType([
StructField("id", IntegerType(), True),
StructField("price", IntegerType(), True),
StructField("time", LongType(), True),
StructField("highorlow", StringType(), True)
])

In [29]:
df_raw = spark.read.json(path, multiLine =True)
df = df_raw.select('data.*')
df2 = df.withColumn(
    "tab",
    sf.array(*[
        sf.struct(
            sf.lit(x).cast("int").alias("id"),
            sf.col(x).alias("count")
            ).alias(x)
        for x in df.columns]
    ),
).selectExpr("inline(tab)")

df3 = df2.select('id', sf.inline(sf.array('count')))

In [ ]:
df_raw.show()

+--------------------+
|                data|
+--------------------+
|{{290, 1776281201...|
+--------------------+



In [ ]:
df.show()

+--------------------+--------------------+--------------------+
|                   2|                   6|                   8|
+--------------------+--------------------+--------------------+
|{290, 1776281201,...|{187838, 17762807...|{195680, 17762808...|
+--------------------+--------------------+--------------------+



In [ ]:
df2.show()

+---+--------------------+
| id|               count|
+---+--------------------+
|  2|{290, 1776281201,...|
|  6|{187838, 17762807...|
|  8|{195680, 17762808...|
+---+--------------------+

